In [ ]:
import numpy as np
from activators import ReluActivator,IdentityActivator

# **辅助函数**

In [ ]:
def get_patch(input_array,i,j,filter_width,filter_height,stride):
    '''
    :param input_array: 输入数组
    :param i: 图像上的第i行
    :param j: 图像上的第j列
    :param filter_width: 卷积核的宽
    :param filter_height: 卷积核的高
    :param stride: 步长
    :return: 切出的方块
    '''
    # 小方块左上角的坐标
    start_i=i*stride
    start_j=j*stride
    # 如果是单通道图片：
    if input_array.ndim==2:
        return input_array[start_i:start_i+filter_height,start_j:start_j+filter_width]
#     如果是多通道图片：
    elif input_array.ndim==3:
        return input_array[:,start_i : start_i + filter_height,start_j : start_j + filter_width]
def get_max_index(array):
    '''
    输入一个二维数组，用于最大池化，找最大值位置并返回坐标
    '''
    max_i=0
    max_j=0
    max_value=array[0,0]

    for i in range(array.shape[0]):
        for j in range(array.shape[1]):
            if array[i,j]>max_value:
                max_value=array[i,j]
                max_i=i
                max_j=j
    return max_i,max_j
def conv(input_array,kernel_array,output_array,stride,bias):
    '''
    :param input_array: 做好padding的输入数组
    :param kernel_array:卷积核权重
    :param output_array:输出数组
    :param stride:步长
    :param bias:偏置
    '''
    channel_number=input_array.ndim
    output_width=output_array.shape[1]
    output_height=output_array.shape[0]

    kernel_width=kernel_array.shape[-1]
    kernel_height=kernel_array.shape[-2]

    for i in range(output_height):
        for j in range(output_width):
            patch=get_patch(input_array,i,j,kernel_width,kernel_height,stride)
            sum_value=(patch*kernel_array).sum()
            output_array[i][j]=sum_value+bias
def padding(input_array,zp):
    '''
    给数组四周补0，zp是圈数，返回补充后的数组，补充后宽高分别增加2zp
    '''
    if zp==0:
        return input_array
    if input_array.ndim==3:
        input_depth=input_array.shape[0]
        input_height=input_array.shape[1]
        input_width=input_array.shape[2]

        padded_array=np.zeros([input_depth,input_height+2*zp,input_width+2*zp])
        padded_array[:,zp:zp+input_height,zp:zp+input_width]=input_array
        return padded_array
    elif input_array.ndim==2:
        input_height=input_array.shape[0]
        input_width=input_array.shape[1]
        padded_array=np.zeros([input_height+2*zp,input_width+2*zp])
        padded_array[zp:zp+input_height,zp:zp+input_width]=input_array
        return padded_array
def element_wise_op(array,op):
    '''
    :param op: 操作
    对数组每个元素应用op函数
    '''
    # np.nditer是迭代器，op_flags=['readwrite']表示可以读取修改元素，i[...]是修改元素的标准写法
    for i in np.nditer(array,op_flags=['readwrite']):
        i[...]=op(i)


# **卷积核类**

In [ ]:
# 卷积核包含权重(权重形状(depth, height, width))和偏置
class Filter(object):
    def __init__(self,width,height,depth):
        self.weights=np.random.uniform(-1e-4,1e-4,(depth,height,width))
        # 初始化权重要小随机数，避免梯度消失或梯度爆炸

        self.bias=0
        self.weights_grad=np.zeros(self.weights.shape)
        self.bias_grad=0
    def update(self,learning_rate):
        '''
        W_new = W_old - η × (∂E/∂W)
        b_new = b_old - η × (∂E/∂b)
        '''
        self.weights-=learning_rate * self.weights_grad
        self.bias-=learning_rate * self.bias_grad






# 卷积层类

In [ ]:
class ConvLayer(object):
    def __init__(self,input_width,input_height,channel_number,filter_width,filter_height,filter_number,zero_padding,stride,activator,learning_rate):
        '''
        :param input_width: 输入图像的宽度
        :param input_height: 输入图像的高度
        :param channel_number: 输入通道数（RGB=3，灰度图像=1）
        :param filter_width: 卷积核尺寸宽
        :param filter_height: 卷积核尺寸高
        :param filter_number: 卷积核数量（输出通道数）
        :param zero_padding: 零填充大小
        :param stride: 步长
        :param activator: 激活函数对象
        :param learning_rate: 学习率
        '''
        self.input_width=input_width
        self.input_height=input_height
        self.channel_number=channel_number
        self.filter_width=filter_width
        self.filter_height=filter_height
        self.filter_number=filter_number
        self.zero_padding=zero_padding
        self.stride=stride




        self.output_width=self.calculate_output_size(self.input_width,filter_width,zero_padding,stride)
        self.output_height=self.calculate_output_size(self.input_height,filter_height,zero_padding,stride)

        self.output_array=np.zeros([self.filter_number,self.output_height,self.output_width])

        self.filters=[]
        for i in range(filter_number):
            self.filters.append(Filter(filter_width,filter_height,self.channel_number))

        self.activator=activator
        self.learning_rate=learning_rate
    @staticmethod
    def calculate_output_size(input_size,filter_size,zero_padding,stride):
        return (input_size - filter_size + 2 * zero_padding) // stride + 1


    def forward(self,input_array):
        # 先添加零填充，再对每个卷积核执行卷积，最后对每个元素应用激活函数正向传播
        self.input_array=input_array
        self.padded_input_array=padding(input_array,self.zero_padding)
        for f in range(self.filter_number):
            filter=self.filters[f]
            conv(self.padded_input_array,filter.get_weights(),self.output_array[f],self.stride,filter.get_bias())
        element_wise_op(self.output_array,self.activator.forward)
    def backward(self,input_array,sensitivity_array,activator):
        '''
        :param input_array: 本层的输入
        :param sensitivity_array: 本层的误差项，这是由后一层传过来的
        :param activator: 上一层的激活函数
        任务：计算要传给前一层的误差项，并计算本层权重的梯度∂E/∂W^l 和 ∂E/∂b^l
        '''
        # 先前向传播得到本层输出
        self.forward(input_array)
        # 计算传递给前一层的误差项
        self.bp_sensitivity_map(sensitivity_array,activator)

        self.bp_gradient(sensitivity_array)

    def expand_sensitivity_map(self,sensitivity_array):
        '''
        在反向传播时，输出位置的误差项只对应输入的一个位置。
        扩展规则：把误差项放在扩展后的数组的 stride 整数倍位置，其他位置填0。
        '''
        depth=sensitivity_array.shape[0]
    #     拓展后的尺寸即为stride=1时的尺寸
        expanded_width=(self.input_width-self.filter_width+2*self.zero_padding+1)
        expanded_height=(self.input_height-self.filter_height+2*self.zero_padding+1)
        expand_array=np.zeros((depth,expanded_height,expanded_width))
        for i in range(self.output_height):
            for j in range(self.output_width):
                i_pos=i*self.stride
                j_pos=j*self.stride
                expand_array[:,i_pos,j_pos]=sensitivity_array[:,i,j]
        return expand_array
     def create_delta_array(self):
        '''创建与输入尺寸相同的全0数组，用于存储δ^{l-1}'''
        return np.zeros((self.channel_number, self.input_height, self.input_width))

    def bp_sensitivity_map(self,sensitivity_array,activator):
        '''
        sensitivity_array: 本层的误差项 δ^l（形状：filter_number × H_out × W_out）
        activator: 上一层的激活函数
        δ^{l-1} = δ^l ⊗ rot180(W^l) ⊙ f'(net^{l-1})
        ⊗ 表示 full 卷积（不是valid卷积）
        rot180 表示把卷积核旋转180度
        ⊙ 表示逐元素相乘
        f'(net^{l-1}) 是上一层的激活函数导数
        '''
        expanded_array=self.expand_sensitivity_map(sensitivity_array)

        # full卷积的输出尺寸 = input_size + filter_size - 1
        # 计算需要的填充大小
        expanded_width = expanded_array.shape[2]
        # 公式：zp = (input_width + filter_width - 1 - expanded_width) / 2
        zp = (self.input_width + self.filter_width - 1 - expanded_width) // 2
        padded_array = padding(expanded_array, zp)

        # delta_array就是 δ^{l-1}，形状和输入 a^{l-1} 相同
        self.delta_array = self.create_delta_array()

        # 公式：δ^{l-1} = Σ_f (δ^l_f ⊗ rot180(W^l_f))
        for f in range(self.filter_number):
            filter = self.filters[f]

            # 把卷积核旋转180度：rot180(W^l)
            # np.rot90(i, 2) 旋转90度两次 = 180度
            # 用列表推导式对每个通道做旋转
            flipped_weights = np.array([np.rot90(i, 2) for i in filter.get_weights()])

            # 计算这个卷积核贡献的delta_array
            delta_array = self.create_delta_array()
            for d in range(delta_array.shape[0]):  # d是通道索引
                # full卷积：padded_array[f] 是 δ^l_f
                # flipped_weights[d] 是旋转后的卷积核
                # 结果加到delta_array[d]
                conv(padded_array[f], flipped_weights[d],
                     delta_array[d], 1, 0)  # stride=1，bias=0
            self.delta_array += delta_array

        # 公式中的 ⊙ f'(net^{l-1})
        derivative_array = np.array(self.input_array)  # 复制一份
        # 对每个元素应用activator.backward（激活函数的导数）
        element_wise_op(derivative_array, activator.backward)
        self.delta_array *= derivative_array

    def bp_gradient(self,sensitivity_array):
        expanded_array = self.expand_sensitivity_map(sensitivity_array)
        for f in range(self.filter_number):
            filter = self.filters[f]
            # 计算每个权重的梯度：∂E/∂W^l
            # 对每个输入通道d
            for d in range(filter.weights.shape[0]):  # shape[0]是输入通道数
                # valid卷积：padded_input_array[d] 是输入的第d个通道
                # expanded_array[f] 是误差项的第f个通道
                # 结果存入filter.weights_grad[d]
                conv(self.padded_input_array[d],
                     expanded_array[f],
                     filter.weights_grad[d], 1, 0)

            # 计算偏置的梯度：∂E/∂b^l
            # 对所有位置的误差项求和
            filter.bias_grad = expanded_array[f].sum()





